# Reporte de bajo rendimiento académico

Estudiantes con **Nota final < 3.0** agrupados por programa, semestre, sede y asignatura.

In [46]:
import pandas as pd

notas = pd.read_csv("../data/raw/notas_pivot.csv")
est = pd.read_csv("../data/raw/estudiantes.csv")

df = notas.merge(
    est[["Numero_identificacion", "Nombre_nivel", "Sede", "Nombre_programa_limpio"]],
    left_on="Numero_identificacion_estudiante",
    right_on="Numero_identificacion",
    how="left"
)

df["bajo"] = (df['Nota final'] < 3.0) & (df['Nota final'] > 1)
df["condición_de_alerta"] = (df['Nota final'] <=1) & (df['Nota final'] >=0)

print(f"Total registros: {len(df):,}")
print(f"registros con nota < 3.0: {df['bajo'].sum():,}")
print(f"registos con condición de alerta: {df['condición_de_alerta'].sum():,}")

Total registros: 5,572
registros con nota < 3.0: 294
registos con condición de alerta: 281


In [47]:
import unicodedata
# Clasificacion por area
def _asignar_area(nombre):
    nombre = str(nombre).strip()
    nombre = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode("ascii").upper()
    m = {
        "Logistica y Comercio": [
            "GESTION NAVIERA Y PORTUARIA",
            "INTRODUCCION AL COMERCIO EXTERIOR",
            "INTRODUCCION A LA LOGISTICA",
            "INFRAESTRUCTURA PORTUARIA",
            "LOGISTICA COMERCIAL",
            "LEGISLACION COMERCIAL Y ADUANERA",
        ],
        "Marketing y Ventas": [
            "INTRODUCCION AL MARKETING DIGITAL",
            "GESTION DE PRECIOS Y PERCEPCION DE VALOR",
            "SEO Y SEM POSICIONAMIENTO WEB",
            "ESTRATEGIAS DE CONTENIDO EN REDES SOCIALES",
        ],
        "Turismo": [
            "ORGANIZACION DE SERVICIOS EN AGENCIAS DE VIAJES",
            "ETNOTURISMO Y DESARROLLO COMUNITARIO",
            "SOSTENIBILIDAD Y TURISMO REGENERATIVO",
            "LEGISLACION EN LA INDUSTRIA TURISTICA",
            "ETIQUETA Y PROTOCOLO EN SERVICIOS TURISTICOS",
            "FUNDAMENTOS DE LA OPERACION TURISTICA",
        ],
        "Idiomas": [
            "INGLES I",
            "INGLES II",
        ],
        "Matematicas y Estadistica": [
            "RAZONAMIENTO CUANTITATIVO",
            "PROBABILIDAD Y ESTADISTICA",
            "ESTADISTICA",
            "ALGEBRA Y GEOMETRIA ANALITICA",
            "CALCULO DIFERENCIAL",
        ],
        "Comunicacion": [
            "COMUNICACION ESCRITA",
            "EXPRESION ORAL Y ARGUMENTACION",
            "LECTOESCRITURA Y COMUNICACION TECNICA",
        ],
        "Formacion Integral": [
            "ETICA PROFESIONAL EN LOS NEGOCIOS",
            "COMPETENCIAS HUMANAS Y DESARROLLO INTEGRAL",
            "CATEDRA USM Y CONTEXTO SAMARIO",
            "TALLER DE CREATIVIDAD E INNOVACION",
        ],
        "Tecnologia y Datos": [
            "HERRAMIENTAS TICS",
            "HERRAMIENTAS TECNOLOGICAS APLICADAS I",
            "MINERIA DE DATOS I",
        ],
        "Diseno y Modas": [
            "PATRONAJE Y ESCALADO",
            "TECNICAS DE CONFECCION I",
            "PRODUCCION DE MODAS",
            "HISTORIA DE LA MODA E INDUMENTARIA",
        ],
        "Gastronomia": [
            "CULTURA Y GASTRONOMIA",
            "MIXOLOGIA Y COCTELERIA",
            "GASTRONOMIA II",
            "SST Y BPM",
            "GASTRONOMIA I",
            "BIOQUIMICA",
        ],
        "Administracion y Costos": [
            "ADMINISTRACION DE ORGANIZACIONES",
            "COSTOS Y PRESUPUESTOS",
        ],
    }
    for area, materias in m.items():
        if nombre in materias:
            return area
    return "Otra"

df["Area"] = df["Nombre_asignatura"].apply(_asignar_area)
print(df["Area"].value_counts())

Area
Matematicas y Estadistica    876
Comunicacion                 803
Idiomas                      736
Logistica y Comercio         582
Tecnologia y Datos           575
Turismo                      462
Gastronomia                  438
Formacion Integral           381
Marketing y Ventas           380
Diseno y Modas               272
Administracion y Costos       67
Name: count, dtype: int64


In [48]:
print(df.columns)
df['Nombre_curso'].nunique()
#df['bajo']


Index(['Consecutivo_curso', 'Nombre_curso', 'Codigo_asignatura',
       'Nombre_asignatura', 'Codigo_programa', 'Nombre_programa',
       'Consecutivo_periodo', 'Nombre_periodo', 'Consecutivo_sede_jornada',
       'Nombre_sede_jornada', 'Nombre_completo_docente',
       'Numero_identificacion_docente', 'Codigo_matricula',
       'Numero_identificacion_estudiante',
       'Abreviatura_tipo_identificacion_estudiante',
       'Nombre_completo_estudiante', 'Promedio_evaluacion',
       'Porcentaje_evaluado', 'Cantidad_inasistencia',
       'Porcentaje_inasistencia', 'Estudiante_formalizado', 'Observaciones',
       'Primer Seguimiento', 'Segundo Seguimiento', 'Tercer Seguimiento',
       'Grupo', 'Nota final', 'Numero_identificacion', 'Nombre_nivel', 'Sede',
       'Nombre_programa_limpio', 'bajo', 'condición_de_alerta', 'Area'],
      dtype='str')


185

In [49]:
tabla_resumen_bajorendimiento = df.groupby(["Nombre_curso","Sede",'Nombre_programa_limpio','Grupo']).agg(
    Total_estudiantes=("bajo", "count"),
    Bajo_rendimiento=("bajo", "sum"),
    Porcentaje_bajo=("bajo", "mean")
).reset_index()
tabla_resumen_bajorendimiento["Porcentaje_bajo"] = (tabla_resumen_bajorendimiento["Porcentaje_bajo"] * 100).round(1)

tabla_resumen_bajorendimiento = tabla_resumen_bajorendimiento.sort_values("Porcentaje_bajo", ascending=False).reset_index(drop=True)

display(tabla_resumen_bajorendimiento[tabla_resumen_bajorendimiento['Porcentaje_bajo']>0])

,Nombre_curso,Sede,Nombre_programa_limpio,Grupo,Total_estudiantes,Bajo_rendimiento,Porcentaje_bajo
0,261-BAS-42010107-ÉTICA PROFESIONAL EN LOS NEGO...,BASTIDAS,TECNOLOGÍA EN MARKETING DIGITAL,A,29,7,24.1
1,261-BAS-42040107-GESTIÓN NAVIERA Y PORTUARIA,BASTIDAS,TECNOLOGÍA EN GESTIÓN DE OPERACIONES LOGÍSTICAS,A,35,8,22.9
2,261-PES-42010104-INGLÉS I,PESCAITO,TECNOLOGÍA EN MARKETING DIGITAL,A,42,9,21.4
3,261-INE-42030101-COMUNICACIÓN ESCRITA (A),INEM,TECNOLOGÍA EN GESTIÓN DE SERVICIOS GASTRONÓMICOS,A,33,7,21.2
4,261-INE-42010104-INGLÉS I (A),INEM,TECNOLOGÍA EN MARKETING DIGITAL,A,49,10,20.4
...,...,...,...,...,...,...,...
96,261-INE-42030103-RAZONAMIENTO CUANTITATIVO (C),INEM,TECNOLOGÍA EN GESTIÓN DE SERVICIOS GASTRONÓMICOS,C,40,1,2.5
97,261-INE-42030102-HERRAMIENTAS TICS (C),INEM,TECNOLOGÍA EN GESTIÓN DE SERVICIOS GASTRONÓMICOS,C,42,1,2.4
98,261-PES-42010107-ÉTICA PROFESIONAL EN LOS NEGO...,PESCAITO,TECNOLOGÍA EN MARKETING DIGITAL,A,42,1,2.4
99,261-PES-42010106-INTRODUCCIÓN AL MARKETING DIG...,PESCAITO,TECNOLOGÍA EN MARKETING DIGITAL,A,42,1,2.4


In [50]:
print("=== TOP 10 ASIGNATURAS CON MAYOR BAJO RENDIMIENTO ===")
display(tabla_resumen_bajorendimiento.head(10))

=== TOP 10 ASIGNATURAS CON MAYOR BAJO RENDIMIENTO ===


,Nombre_curso,Sede,Nombre_programa_limpio,Grupo,Total_estudiantes,Bajo_rendimiento,Porcentaje_bajo
0,261-BAS-42010107-ÉTICA PROFESIONAL EN LOS NEGO...,BASTIDAS,TECNOLOGÍA EN MARKETING DIGITAL,A,29,7,24.1
1,261-BAS-42040107-GESTIÓN NAVIERA Y PORTUARIA,BASTIDAS,TECNOLOGÍA EN GESTIÓN DE OPERACIONES LOGÍSTICAS,A,35,8,22.9
2,261-PES-42010104-INGLÉS I,PESCAITO,TECNOLOGÍA EN MARKETING DIGITAL,A,42,9,21.4
3,261-INE-42030101-COMUNICACIÓN ESCRITA (A),INEM,TECNOLOGÍA EN GESTIÓN DE SERVICIOS GASTRONÓMICOS,A,33,7,21.2
4,261-INE-42010104-INGLÉS I (A),INEM,TECNOLOGÍA EN MARKETING DIGITAL,A,49,10,20.4
5,261-BAS-42040102-HERRAMIENTAS TICS,BASTIDAS,TECNOLOGÍA EN GESTIÓN DE OPERACIONES LOGÍSTICAS,A,35,7,20.0
6,261-INE-42040104-INGLÉS I (B),INEM,TECNOLOGÍA EN GESTIÓN DE OPERACIONES LOGÍSTICAS,B,37,7,18.9
7,261-BAS-42040105-INTRODUCCIÓN A LA LOGÍSTICA,BASTIDAS,TECNOLOGÍA EN GESTIÓN DE OPERACIONES LOGÍSTICAS,A,35,6,17.1
8,261-INE-42030209-INGLÉS II,INEM,TECNOLOGÍA EN GESTIÓN DE SERVICIOS GASTRONÓMICOS,A,36,6,16.7
9,261-PES-42040102-HERRAMIENTAS TICS,PESCAITO,TECNOLOGÍA EN GESTIÓN DE OPERACIONES LOGÍSTICAS,A,45,7,15.6


In [51]:
tabla_resumen_bajorendimiento_asigantura = df.groupby(['Nombre_asignatura']).agg(
    Total_estudiantes=("bajo", "count"),
    Bajo_rendimiento=("bajo", "sum"),
    Porcentaje_bajo=("bajo", "mean")
).reset_index()
tabla_resumen_bajorendimiento_asigantura["Porcentaje_bajo"] = (tabla_resumen_bajorendimiento_asigantura["Porcentaje_bajo"] * 100).round(1)

tabla_resumen_bajorendimiento_asigantura = tabla_resumen_bajorendimiento_asigantura.sort_values("Porcentaje_bajo", ascending=False).reset_index(drop=True)

display(tabla_resumen_bajorendimiento_asigantura[tabla_resumen_bajorendimiento_asigantura['Bajo_rendimiento']>0])

,Nombre_asignatura,Total_estudiantes,Bajo_rendimiento,Porcentaje_bajo
0,INGLÉS I,525,64,12.2
1,MINERÍA DE DATOS I,19,2,10.5
2,MIXOLOGÍA Y COCTELERÍA,36,3,8.3
3,GASTRONOMÍA II,36,3,8.3
4,INTRODUCCIÓN A LA LOGÍSTICA,163,13,8.0
5,HERRAMIENTAS TICS,525,42,8.0
6,GESTIÓN NAVIERA Y PORTUARIA,164,12,7.3
7,GASTRONOMÍA I,110,8,7.3
8,ÉTICA PROFESIONAL EN LOS NEGOCIOS,177,13,7.3
9,HERRAMIENTAS TECNOLÓGICAS APLICADAS I,31,2,6.5


In [52]:
asignaturas= df['Nombre_asignatura'].unique()
asignaturas


<ArrowStringArray>
[                    'GESTIÓN NAVIERA Y PORTUARIA',
               'INTRODUCCIÓN AL COMERCIO EXTERIOR',
                     'INTRODUCCIÓN A LA LOGÍSTICA',
                                        'INGLÉS I',
                       'RAZONAMIENTO CUANTITATIVO',
                               'HERRAMIENTAS TICS',
                            'COMUNICACIÓN ESCRITA',
               'ÉTICA PROFESIONAL EN LOS NEGOCIOS',
               'INTRODUCCIÓN AL MARKETING DIGITAL',
                      'PROBABILIDAD Y ESTADÍSTICA',
 'ORGANIZACIÓN DE SERVICIOS EN AGENCIAS DE VIAJES',
            'ETNOTURISMO Y DESARROLLO COMUNITARIO',
           'SOSTENIBILIDAD Y TURISMO REGENERATIVO',
           'LEGISLACIÓN EN LA INDUSTRIA TURÍSTICA',
                                       'INGLÉS II',
                  'EXPRESIÓN ORAL Y ARGUMENTACIÓN',
    'ETIQUETA Y PROTOCOLO EN SERVICIOS TURÍSTICOS',
           'FUNDAMENTOS DE LA OPERACIÓN TURÍSTICA',
                                     'ESTADÍS

In [ ]:
tabla_resumen_bajorendimiento_area = df.groupby(['Area']).agg(
    Total_estudiantes=("bajo", "count"),
    Bajo_rendimiento=("bajo", "sum"),
    Porcentaje_bajo=("bajo", "mean")
).reset_index()
tabla_resumen_bajorendimiento_area["Porcentaje_bajo"] = (tabla_resumen_bajorendimiento_area["Porcentaje_bajo"] * 100).round(1)

tabla_resumen_bajorendimiento_area = tabla_resumen_bajorendimiento_area.sort_values("Porcentaje_bajo", ascending=False).reset_index(drop=True)

display(tabla_resumen_bajorendimiento_area[tabla_resumen_bajorendimiento_area['Bajo_rendimiento']>0])

,Area,Total_estudiantes,Bajo_rendimiento,Porcentaje_bajo
0,Idiomas,736,77,10.5
1,Tecnologia y Datos,575,46,8.0
2,Gastronomia,438,26,5.9
3,Logistica y Comercio,582,32,5.5
4,Comunicacion,803,39,4.9
5,Matematicas y Estadistica,876,36,4.1
6,Formacion Integral,381,15,3.9
7,Administracion y Costos,67,2,3.0
8,Marketing y Ventas,380,11,2.9
9,Turismo,462,10,2.2


In [ ]:
df["Segundo Seguimiento"] = df.apply(
    lambda r: None if r["Grupo"] == "B" else r["Segundo Seguimiento"], axis=1
)
df["Tercer Seguimiento"] = df.apply(
    lambda r: None if r["Grupo"] == "B" else r["Tercer Seguimiento"], axis=1
)
df["Nota final"] = df.apply(
    lambda r: r["Primer Seguimiento"] if r["Grupo"] == "B"
    else (r.get("Primer Seguimiento") or 0) * 0.3
    + (r.get("Segundo Seguimiento") or 0) * 0.3
    + (r.get("Tercer Seguimiento") or 0) * 0.4,
    axis=1
)

Estudiantes_revision = df[df['condición_de_alerta'] == True][
    ['Nombre_completo_estudiante', 'Sede', 'Nombre_programa_limpio', 'Grupo',
     'Primer Seguimiento', 'Segundo Seguimiento', 'Tercer Seguimiento', 'Nota final']
].drop_duplicates()
display(Estudiantes_revision)


,Nombre_completo_estudiante,Sede,Nombre_programa_limpio
37,RODRIGUEZ DE AGUAS ESTEBAN DAVID,PESCAITO,TECNOLOGÍA EN GESTIÓN DE OPERACIONES LOGÍSTICAS
49,Ascanio Vaca Liseth Lorena,PESCAITO,TECNOLOGÍA EN GESTIÓN DE OPERACIONES LOGÍSTICAS
69,Medina Caro Rafael Enrique,PESCAITO,TECNOLOGÍA EN GESTIÓN DE OPERACIONES LOGÍSTICAS
155,Lopez Dzgranados Moises Alexander,PESCAITO,TECNOLOGÍA EN GESTIÓN DE OPERACIONES LOGÍSTICAS
216,RODRIGUEZ AVENDAÑO LAIMA JAZAI,PESCAITO,TECNOLOGÍA EN GESTIÓN DE OPERACIONES LOGÍSTICAS
...,...,...,...
5420,PULIDO PARRA LUZ ANGELA,BASTIDAS,TECNOLOGÍA EN MARKETING DIGITAL
5430,AYOLA ADARRAGA DANIEL ELIAS,BASTIDAS,TECNOLOGÍA EN MARKETING DIGITAL
5440,Guevara Perez Geovanis Junior,BASTIDAS,TECNOLOGÍA EN MARKETING DIGITAL
5446,NUÑEZ PADILLA STEVAN DAVID,BASTIDAS,TECNOLOGÍA EN MARKETING DIGITAL
